In [12]:
import numpy as np
import torch
import mne

from src.preprocess import EEGPreprocessor
from src.features import extract_band_powers
from src.model import EEGNetLight

np.random.seed(42)
torch.manual_seed(42)


sfreq = 250.0
n_channels = 32
n_samples = int(120 * sfreq)
ch_names = [f"EEG{i+1:03d}" for i in range(n_channels)]
info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="eeg")

times = np.arange(n_samples) / sfreq
raw_data = np.random.randn(n_channels, n_samples) * 1e-5
raw_data += 0.5e-5 * np.sin(2 * np.pi * 50 * times)  # نویز ۵۰Hz
raw_data[0:4, :] += np.sin(2 * np.pi * 0.5 * times) ** 8 * 4e-5  # نویز پلک

raw = mne.io.RawArray(raw_data, info)
print(f"Data loaded: {raw.info}")


preprocessor = EEGPreprocessor(l_freq=1.0, h_freq=40.0, notch_freq=50.0)
filtered_raw = preprocessor.apply_filters(raw)
ica = preprocessor.fit_ica(filtered_raw, n_components=10)
cleaned_raw = preprocessor.remove_eog_artifacts(filtered_raw, ica)
print("Preprocessing & ICA completed successfully.")


features = extract_band_powers(cleaned_raw)
for band, power in features.items():
    print(f"Band: {band:5s} | Mean Power across channels: {np.mean(power):.4e}")


n_samples_model = 128
model = EEGNetLight(num_channels=n_channels, num_classes=2, samples=n_samples_model)
model.eval()

dummy_input = torch.randn(8, 1, n_channels, n_samples_model)
with torch.no_grad():
    predictions = model(dummy_input)

print(f"\nEEGNet forward pass output shape: {predictions.shape}")
print("🎉 All pipeline stages executed successfully with 0 errors!")



Creating RawArray with float64 data, n_channels=32, n_times=30000
    Range : 0 ... 29999 =      0.000 ...   119.996 secs
Ready.
Data loaded: <Info | 7 non-empty values
 bads: []
 ch_names: EEG001, EEG002, EEG003, EEG004, EEG005, EEG006, EEG007, EEG008, ...
 chs: 32 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 125.0 Hz
 meas_date: unspecified
 nchan: 32
 projs: []
 sfreq: 250.0 Hz
>
Fitting ICA to data using 32 channels (please be patient, this may take a while)
Selecting by number: 10 components
Fitting ICA took 4.5s.
Applying ICA to Raw instance
    Transforming to ICA space (10 components)
    Zeroing out 0 ICA components
    Projecting back using 32 PCA components
Preprocessing & ICA completed successfully.
Effective window size : 8.192 (s)
Band: Delta | Mean Power across channels: 7.8533e-12
Band: Theta | Mean Power across channels: 7.9473e-13
Band: Alpha | Mean Power across channels: 7.8754e-13
Band: Beta  | Mean Power across channels: 8.0379e-13
Band: Gamma | Mean 

c:\Users\avina\miniconda3\envs\cross\lib\site-packages\sklearn\decomposition\_fastica.py:128: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
